# Infinite-Chain FHMM — IBP-FHMM with Gibbs Sampling

A **Factorial HMM** with an unknown number of chains places a binary **activity matrix**
$Z \in \{0,1\}^{T \times K}$ over the contributions of $K$ parallel Markov chains:

$$y(t) = \sum_{k=1}^{K} Z_{t,k}\,\mu_k[x_k(t)] + \varepsilon(t)$$

Each chain $x_k$ follows its own Markov dynamics with transition matrix $A_k$ and initial
distribution $\pi_k$. The activity entries $Z_{t,k}$ determine which chains are “switched on”
at each timestep, and receive an **Indian Buffet Process** (IBP) prior:

$$p(Z_{t,k} = 1 \mid Z_{-t,k}) = \frac{m_{-t,k}}{T}, \qquad m_{-t,k} = \sum_{t' \neq t} Z_{t',k}$$

New chains are proposed each iteration by drawing $m_{\mathrm{new}} \sim \mathrm{Poisson}(\alpha/T)$
and appending them fully active. `InfiniteFHMMGibbs` alternates five Gibbs steps:

- **Step 1** — Sample $Z_{t,k}$ from its conditional posterior (IBP prior × Gaussian likelihood ratio).
- **Step 2** — Sample $x_k$ via **Forward Filtering Backward Sampling** (FFBS).
- **Step 3** — Update emission parameters by MLE on active residuals.
- **Step 4** — Prune chains with no active timesteps.
- **Step 5** — Propose new chains via $\mathrm{Poisson}(\alpha/T)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from InfiniteFHMMGibbs import InfiniteFHMMGibbs

## 1. Generating Synthetic Data

We simulate three two-state Markov chains with sticky transitions (stay probability 0.97).
All chains share the same emission structure: state 0 emits from $\mathcal{N}(0, 0.05)$ and
state 1 from $\mathcal{N}(1.5, 0.2)$. Their noisy emissions are summed to produce the single
mixed observation $y(t)$ that the sampler will see.

The model receives only $y(t)$ and must automatically discover the number of chains and
recover their emission parameters.

In [ ]:
np.random.seed(20012025)

T = 1000
true_chains = 3
mu_true = np.array([0.0, 1.5])
var_true = np.array([0.05, 0.2])

hidden = np.zeros((T, true_chains), dtype=int)
chain_signals = np.zeros((T, true_chains))
obs = np.zeros(T)

for c in range(true_chains):
    z = np.zeros(T, dtype=int)
    z[0] = np.random.choice(2, p=[0.5, 0.5])
    for tt in range(1, T):
        p = [0.97, 0.03] if z[tt - 1] == 0 else [0.03, 0.97]
        z[tt] = np.random.choice(2, p=p)
    hidden[:, c] = z
    chain_signals[:, c] = mu_true[z]
    obs += mu_true[z] + np.random.normal(0, np.sqrt(var_true[z]))

t = np.arange(T)
print('True emission means:', mu_true)
print('True emission vars: ', var_true)
print(f'Observation range:  [{obs.min():.2f}, {obs.max():.2f}]')

## 2. Individual Chain Signals

Each chain $k$ emits $\mu_k[x_k(t)]$ at every timestep — a step function alternating
between the two emission levels as the hidden state switches. The shaded regions mark
periods where the chain is in state 1 (high-emission). These latent step functions are
what the sampler will attempt to recover from the mixed observation.

In [ ]:
COLORS = ['#2196F3', '#4CAF50', '#FF5722']

fig, axes = plt.subplots(true_chains, 1, figsize=(12, 5), sharex=True)
for c, ax in enumerate(axes):
    ax.fill_between(t, 0, 1, where=(hidden[:, c] == 1),
                    transform=ax.get_xaxis_transform(),
                    color=COLORS[c], alpha=0.18)
    ax.plot(t, chain_signals[:, c], color=COLORS[c], lw=1.5)
    ax.set_ylabel(f'Chain {c}', fontsize=10)
    lo, hi = chain_signals.min() - 0.05, chain_signals.max() + 0.05
    ax.set_ylim(lo, hi)

axes[0].set_title(r'True individual chain signals  $\mu_k[x_k(t)]$', fontsize=12)
axes[-1].set_xlabel('Time step $t$')
plt.tight_layout()
plt.show()

## 3. The Mixed Observation

The sampler receives only $y(t)$ — the noisy superposition of all chain emissions. From this
single trace it must infer how many chains are present, which timesteps each chain is active,
and what each chain's emission parameters are.

In [ ]:
noiseless = chain_signals.sum(axis=1)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, obs, color='#555', lw=1, alpha=0.7, label='Observed $y(t)$')
ax.plot(t, noiseless, color='#E91E63', lw=1.5, ls='--',
        label=r'Noiseless sum  $\sum_k \mu_k[x_k(t)]$')
ax.set_xlabel('Time step $t$')
ax.set_ylabel('Amplitude')
ax.set_title(r'Mixed observation  $y(t) = \sum_k Z_{t,k}\,\mu_k[x_k(t)] + \varepsilon(t)$')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 4. Running the Gibbs Sampler

We call `gibbs_sample`, which runs `n_iter` iterations of the five Gibbs steps. The sampler
starts with `max_initial_chains` fully active chains and adapts the number automatically
via IBP proposals and pruning.

The returned `Z` matrix records which chains are active at each timestep after the final
iteration, and `viterbi_paths` gives the most-likely state sequence per discovered chain.

In [ ]:
iFHMM = InfiniteFHMMGibbs(alpha=3.0, n_states=2)
iFHMM.initialize(obs, max_initial_chains=4)

Z, X_samples, mus, vars_, viterbi_paths = iFHMM.gibbs_sample(obs, n_iter=200)

K_disc = len(mus)
print(f'Discovered {K_disc} chains  (true: {true_chains})')
print()
for k in range(K_disc):
    order = np.argsort(mus[k])
    active_frac = Z[:, k].mean()
    print(f'  Chain {k}: means={mus[k][order].round(4)},  '
          f'vars={vars_[k][order].round(4)},  active={active_frac:.2f}')
print()
print('True means:', mu_true, '  True vars:', var_true)

## 5. Activity Matrix and Viterbi Paths

The **activity matrix** $Z$ (top panel) shows which chains are switched on at each
timestep — blue cells indicate active ($Z_{t,k}=1$), white cells indicate inactive.

Each lower panel shows the Viterbi-decoded state sequence for one discovered chain;
the light-blue shading marks the timesteps where that chain is active according to
the final sampled $Z$.

In [ ]:
K_disc = len(mus)
chain_colors = (COLORS * ((K_disc // len(COLORS)) + 1))[:K_disc]

fig, axes = plt.subplots(K_disc + 1, 1, figsize=(12, 3 + 2 * K_disc), sharex=True)

ax_z = axes[0]
ax_z.imshow(Z.T, aspect='auto', cmap='Blues', interpolation='nearest',
            extent=[-0.5, T - 0.5, K_disc - 0.5, -0.5])
ax_z.set_ylabel('Chain', fontsize=10)
ax_z.set_title(r'Activity matrix $Z_{t,k}$ and per-chain Viterbi paths', fontsize=12)
ax_z.set_yticks(range(K_disc))

for c in range(K_disc):
    ax = axes[c + 1]
    ax.fill_between(t, 0, 1, where=(Z[:, c] == 1),
                    transform=ax.get_xaxis_transform(),
                    color='#90CAF9', alpha=0.35, label='Active ($Z_{t,k}=1$)')
    ax.step(t, viterbi_paths[c], color=chain_colors[c], lw=1.2,
            where='post', label='Viterbi path')
    ax.set_ylim(-0.15, 1.15)
    ax.set_yticks([0, 1])
    ax.set_ylabel(f'Chain {c}', fontsize=10)
    if c == 0:
        ax.legend(loc='upper right', ncol=2, fontsize=8)

axes[-1].set_xlabel('Time step $t$')
plt.tight_layout()
plt.show()

## 6. Signal Reconstruction

The **reconstructed contribution** of discovered chain $k$ at time $t$ uses the Viterbi
path $\hat{x}_k(t)$ weighted by the activity indicator:

$$\hat{s}_k(t) = Z_{t,k}\,\mu_k[\hat{x}_k(t)]$$

Summing across all discovered chains gives the total reconstruction
$\hat{y}(t) = \sum_k \hat{s}_k(t)$, compared against the noisy observation and the
noiseless ground truth.

In [ ]:
recovered = np.zeros((T, K_disc))
for k in range(K_disc):
    recovered[:, k] = Z[:, k] * mus[k][viterbi_paths[k]]
reconstruction = recovered.sum(axis=1)

fig, axes = plt.subplots(K_disc + 1, 1, figsize=(12, 3 + 2 * K_disc), sharex=True)

for c in range(K_disc):
    ax = axes[c]
    ax.plot(t, recovered[:, c], color=chain_colors[c], lw=1.5,
            label=r'Recovered $\hat{s}_k(t)$')
    ax.set_ylabel(f'Chain {c}', fontsize=10)
    if c == 0:
        ax.legend(loc='upper right', fontsize=8)

ax_tot = axes[-1]
ax_tot.plot(t, obs, color='#aaa', lw=1.0, alpha=0.8, label='Observed $y(t)$')
ax_tot.plot(t, noiseless, color='#555', lw=1.2, ls=':', alpha=0.7,
            label=r'Noiseless sum  $\sum_k \mu_k[x_k(t)]$')
ax_tot.plot(t, reconstruction, color='#E91E63', lw=1.5, ls='--',
            label=r'Reconstruction $\hat{y}(t)$')
ax_tot.set_ylabel('Total', fontsize=10)
ax_tot.legend(loc='upper right', ncol=3, fontsize=8)
ax_tot.set_xlabel('Time step $t$')

axes[0].set_title('Discovered chain contributions and total reconstruction', fontsize=12)
plt.tight_layout()
plt.show()